# Libraries

In [ ]:
# Install
%pip install -q spacy benepar openai sentence-transformers pandas
!python -m spacy download en_core_web_md
import benepar; benepar.download('benepar_en3')

# Setup

### Set up karaage

In [ ]:
from karaage import setup, set_verbose

set_verbose(False)

# The backend and the model profile come from KARAAGE_* environment variables or a .env file
# (see the README). By default: the profile qwen3.5-9b on a local LM Studio.
resources = setup()

### Where to save the outputs?

In [ ]:
# Used in the notebook but not in the karaage package, so we set them here:
output_data_folder = "temp/data" # Leave None to skip saving JSON results
output_html_report = "temp/monitoring" # Leave None to skip saving HTML report

# Input

The author-annotation pipeline below expects an English translation of every
article in the input DB. Run **`translate.py`** in this folder first to
populate the two columns it produces:

- `fulltext_translated_qwen3.5-9b` — body translation (used as `text` below)
- `headline_translated_qwen3.5-9b` — headline translation (not used here, but
  written for downstream tooling)

```bash
python translate.py --db ../../data/validation_dataset_datacenter.db
```

Add `--skip-existing` for incremental runs, or `--limit N` for a smoke test.

In [ ]:
# Load SQLite database
file = "../../data/validation_dataset_datacenter.db"
import sqlite3, pandas as pd
conn = sqlite3.connect(file)
df = pd.read_sql_query("SELECT * FROM articles", conn)
print(f"Loaded {len(df)} documents from {file}")

df.head()

In [ ]:
# Build a batch. Reads the English translation written by translate.py.
TEXT_COL   = "fulltext_translated_qwen3.5-9b"
DOC_ID_COL = "DOC_ID"

batch = [
    {
        "text":    row[TEXT_COL],
        "text_id": str(row[DOC_ID_COL]),
    }
    for _, row in df.iterrows()
    if isinstance(row[TEXT_COL], str) and row[TEXT_COL].strip()
]

print(f"Built batch of {len(batch)} documents.")
if len(batch) < len(df):
    print(f"  Skipped {len(df) - len(batch)} row(s) with empty {TEXT_COL!r} "
          f"(run translate.py to fill them).")

# Preview text
if batch:
    print("Text preview:", batch[0]["text"][:100], "...")

# How to process one document

This function does the following:
- Run the pipeline
- Save the output

In [ ]:

# One function to process a document and save results (or get them in memory if you don't want to save)
# You can call this function in a loop for multiple documents, or adapt it to your needs.
# Note: "result" includes intermediate step data and other stuff (time...)
# The final output is in "result.data".
 
import re
from karaage import run, save

def process_document(text, text_id, resources, output_data_folder=None, output_html_report=None):
    """Process a document with the pipeline and optionally save results.
    Args:
        text (str): The input document text.
        text_id (str): A unique identifier for the document (used in filenames).
        resources: The resources object returned by setup().
        output_data_folder (str, optional): Folder to save JSON results. If None, results won't be saved.
        output_html_report (str, optional): Folder to save HTML report. If None, report won't be saved.
    Returns:
        PipelineResult: The result object returned by the pipeline, containing data, timings, etc.
    """
    print(f"\n# Process text {text_id} --- {len(text):,} characters")
    result = run(text, resources)
    if output_data_folder or output_html_report:
        text_id_sanitized = re.sub(r'[^\w\-]+', '_', text_id)[:50]
        save(
            result,
            data_path=f"{output_data_folder}/{text_id_sanitized}.json"            if output_data_folder else None,
            html_path=f"{output_html_report}/monitoring_{text_id_sanitized}.html" if output_html_report else None,
        )
    return result

In [ ]:
# Test: process one document
process_document(text=batch[9]["text"], text_id=batch[9]["text_id"], resources=resources, output_data_folder=output_data_folder, output_html_report=output_html_report)

# Process batch

In [ ]:
# Print total number of characters in the batch
import time

# Estimate duration based on total characters and a rate of 50 seconds per 1000 characters
def format_duration(total_chars, rate_s_per_kchar=50):
    total_seconds = (total_chars / 1000) * rate_s_per_kchar
    days, remainder = divmod(int(total_seconds), 86400)
    hours, remainder = divmod(remainder, 3600)
    minutes, _ = divmod(remainder, 60)

    parts = []
    if days: parts.append(f"{days} day{'s' if days > 1 else ''}")
    if hours: parts.append(f"{hours} hour{'s' if hours > 1 else ''}")
    parts.append(f"{minutes} minute{'s' if minutes > 1 else ''}")

    return " ".join(parts)

total_chars = sum(len(doc["text"]) for doc in batch)
print(f"Total characters in batch: {total_chars:,} --- Duration estimate: {format_duration(total_chars)} (at 50 s/kchar)")

# Process each document in the batch
t0 = time.time()
count = 0
counting_chars = 0
for doc in batch:
    try:
        process_document(doc["text"], doc["text_id"], resources, output_data_folder, output_html_report)
    except Exception as e:
        print(f"Error processing document {doc['text_id']}: {e}")
    count += 1
    counting_chars += len(doc["text"])
    print(f"\n--- {count}/{len(batch)} texts processed --- On average {(time.time() - t0) / (counting_chars / 1000):.1f} s/kchar ---")

# Print total and average processing times
total_time = time.time() - t0
print(f"\nFinished processing {len(batch)} documents in {total_time:.1f} seconds")
print(f"Average time per document: {total_time / len(batch):.1f} seconds.")
print(f"Average time per 1000 characters: {total_time / (total_chars / 1000):.1f} seconds.")

---

# Optional: Debug rephrasing batch

Note: this is not about a batch of documents, but about a batch of statements to rephrase for a given document

In [ ]:
debug = False # Edit me

if (debug):
    from karaage.rephrasing import batch_debug

    # Pick a document to debug (you can change the index to pick a different one):
    doc = batch[0]

    # Run the pipeline (necessary prerequisite to have the "result" object to debug with)
    result = process_document(doc["text"], doc["text_id"], resources) # Note: here we don't save results

    # Debug a specific batch in the rephrasing step:
    DEBUG_BATCH_INDEX = 1  # change this to the batch you want to inspect
    batch_debug(result.all_statements, result.passages, result.main_point, DEBUG_BATCH_INDEX)